# Export v2 models (EfficientNetB2 + DenseNet121) to ONNX for an in-browser demo
CPU only. Each model is exported with two outputs: tumour probability and the last conv feature map (for CAM).
Exports are verified against Keras on validation patches before use.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tf2onnx', 'onnxruntime', 'onnxconverter-common', 'onnx'], check=True)

In [ ]:
import os, json, time, numpy as np, pandas as pd, tensorflow as tf, keras, onnx, onnxruntime as ort, tf2onnx
from PIL import Image
from sklearn.metrics import roc_auc_score
from onnxconverter_common import float16
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
print(tf.__version__, keras.__version__, onnx.__version__, ort.__version__, tf2onnx.__version__)

def find_file(name):
    for root, dirs, files in os.walk('/kaggle/input'):
        dirs[:] = [d for d in dirs if d not in ('train', 'test')]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)
DATA_DIR = os.path.dirname(find_file('train_labels.csv'))
OUT = '/kaggle/working'
SPEC = {'EfficientNetB2': 'top_activation', 'DenseNet121': 'relu'}

# 2000 validation patches (fixed seed) for verification
val = pd.read_csv(find_file('val_pred_efficientnetb2.csv')).sample(2000, random_state=0)
X = np.stack([np.asarray(Image.open(os.path.join(DATA_DIR, 'train', f'{i}.tif')).convert('RGB')) for i in val['id']]).astype('float32')
y = val['label'].values
print(X.shape, y.mean())

In [ ]:
report, cam_w = {}, {}
for name, layer in SPEC.items():
    m = keras.models.load_model(find_file(f'{name}.keras'), compile=False)
    two = keras.Model(m.inputs, [m.output, m.get_layer(layer).output])
    out = m.get_layer('output')
    k, b = out.get_weights()
    cam_w[name] = {'kernel': k[:, 0].astype(float).round(7).tolist(), 'bias': float(b[0]), 'channels': int(k.shape[0])}

    @tf.function(input_signature=[tf.TensorSpec([None, 96, 96, 3], tf.float32, name='image')])
    def fn(x):
        p, f = two(x, training=False)
        return {'prob': tf.cast(p, tf.float32), 'features': tf.cast(f, tf.float32)}

    t0 = time.time()
    # freeze every variable (incl. Normalization mean/variance) into constants, then convert the graph
    frozen = convert_variables_to_constants_v2(fn.get_concrete_function())
    ins = [t.name for t in frozen.inputs if t.dtype != tf.resource]
    outs_ = [t.name for t in frozen.outputs]
    proto, _ = tf2onnx.convert.from_graph_def(frozen.graph.as_graph_def(), input_names=ins, output_names=outs_, opset=17)
    # give the outputs stable names: rank-2 -> 'prob', rank-4 -> 'features'
    g = proto.graph
    for o in list(g.output):
        rank = len(o.type.tensor_type.shape.dim)
        new = 'prob' if rank == 2 else 'features'
        g.node.append(onnx.helper.make_node('Identity', [o.name], [new]))
        vi = onnx.ValueInfoProto(); vi.CopyFrom(o); vi.name = new
        g.output.remove(o); g.output.append(vi)
    g.input[0].name  # keep the input name
    onnx.checker.check_model(proto)
    print('inputs', [i.name for i in g.input], 'outputs', [o.name for o in g.output])
    p32 = f'{OUT}/{name}_fp32.onnx'
    onnx.save(proto, p32)
    p16 = f'{OUT}/{name}.onnx'
    onnx.save(float16.convert_float_to_float16(proto, keep_io_types=True), p16)
    print(name, 'converted in', round(time.time() - t0), 's |', {p: round(os.path.getsize(p) / 2**20, 1) for p in (p32, p16)})

    ref_p = np.concatenate([two.predict(X[i:i + 250], verbose=0)[0].ravel() for i in range(0, len(X), 250)])
    ref_f = two.predict(X[:8], verbose=0)[1]
    r = {'keras_auc': roc_auc_score(y, ref_p)}
    for tag, path in [('fp32', p32), ('fp16', p16)]:
        s = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
        names = [o.name for o in s.get_outputs()]
        outs = [s.run(None, {s.get_inputs()[0].name: X[i:i + 250]}) for i in range(0, len(X), 250)]
        pi, fi = names.index('prob'), names.index('features')
        p = np.concatenate([o[pi].ravel() for o in outs])
        f8 = s.run(None, {s.get_inputs()[0].name: X[:8]})[fi]
        r[tag] = {'auc': roc_auc_score(y, p), 'max_abs_prob_diff': float(np.abs(p - ref_p).max()),
                  'mean_abs_prob_diff': float(np.abs(p - ref_p).mean()),
                  'decision_flips@0.5': int(((p >= .5) != (ref_p >= .5)).sum()),
                  'feature_max_abs_diff': float(np.abs(f8 - ref_f).max()), 'feature_shape': list(f8.shape),
                  'input': s.get_inputs()[0].name, 'outputs': names}
    report[name] = r
    print(json.dumps(r, indent=1, default=float))

with open(f'{OUT}/cam_weights.json', 'w') as fh:
    json.dump(cam_w, fh)
with open(f'{OUT}/onnx_verification.json', 'w') as fh:
    json.dump(report, fh, indent=1, default=float)

# ensemble check with fp16 models vs Keras (probability average, as in the demo)
print({n: {t: report[n][t]['auc'] for t in ('fp32', 'fp16')} for n in report}, {n: report[n]['keras_auc'] for n in report})
print(sorted(os.listdir(OUT)))